In [ ]:
from nba_api.stats.endpoints import leaguedashlineups
import pandas as pd
from pathlib import Path
import plotly.express as px

In [2]:
RAW_DIR = Path("../data/raw")
PROCESSED_DIR = Path("../data/processed")

RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

KNICKS_TEAM_ID = "1610612752"

## PUll Playoff Lineup Data

In [3]:
playoff_lineups = leaguedashlineups.LeagueDashLineups(
    season="2025-26",
    team_id_nullable=KNICKS_TEAM_ID,
    season_type_all_star="Playoffs",
    measure_type_detailed_defense="Advanced",
    per_mode_detailed="Per100Possessions"
)

playoff_df = playoff_lineups.get_data_frames()[0]

playoff_df.head()

,GROUP_SET,GROUP_ID,GROUP_NAME,TEAM_ID,TEAM_ABBREVIATION,GP,W,L,W_PCT,MIN,...,AST_RATIO_RANK,OREB_PCT_RANK,DREB_PCT_RANK,REB_PCT_RANK,TM_TOV_PCT_RANK,EFG_PCT_RANK,TS_PCT_RANK,PACE_RANK,PIE_RANK,SUM_TIME_PLAYED
0,Lineups,-1626157-1628384-1628404-1628969-1628973-,K. Towns - O. Anunoby - J. Hart - M. Bridges -...,1610612752,NYK,12,10,2,0.833,205.0,...,28,61,54,57,64,43,43,71,59,613930
1,Lineups,-1626157-1628404-1628969-1628973-1630540-,K. Towns - J. Hart - M. Bridges - J. Brunson -...,1610612752,NYK,6,5,1,0.833,32.0,...,29,54,57,57,65,48,49,80,67,94845
2,Lineups,-1626157-1628384-1628404-1628973-1630540-,K. Towns - O. Anunoby - J. Hart - J. Brunson -...,1610612752,NYK,5,4,1,0.800,30.0,...,41,41,60,54,66,36,33,61,33,90625
3,Lineups,-1628404-1628969-1628973-1629011-1630540-,J. Hart - M. Bridges - J. Brunson - M. Robinso...,1610612752,NYK,4,3,1,0.750,22.0,...,45,42,55,55,62,37,40,83,66,65750
4,Lineups,-1626157-1628384-1628969-1628973-1629013-,K. Towns - O. Anunoby - M. Bridges - J. Brunso...,1610612752,NYK,3,3,0,1.000,21.0,...,42,39,37,26,57,18,19,78,13,61790


In [4]:
playoff_df.to_csv(
    RAW_DIR / "knicks_lineups_playoffs_raw_2025_26.csv",
    index=False
)

## Standardize Column Names

In [5]:
playoff_clean = playoff_df.copy()

playoff_clean.columns = (
    playoff_clean.columns
    .str.lower()
    .str.strip()
    .str.replace(" ", "_")
)

In [6]:
playoff_clean["season_type"] = "Playoffs"

## Filter Lineups

In [7]:
playoff_clean = playoff_clean[
    playoff_clean["min"] >= 10
].copy()

In [8]:
playoff_clean.to_csv(
    PROCESSED_DIR / "knicks_lineups_playoffs_clean.csv",
    index=False
)

## Load Regular Season Data

In [9]:
regular_df = pd.read_csv(
    PROCESSED_DIR / "knicks_lineups_clustered.csv"
)

regular_df["season_type"] = "Regular Season"

## Create Playoff Features

In [10]:
playoff_clean["lineup_quality"] = pd.cut(
    playoff_clean["net_rating"],
    bins=[-999, -5, 0, 5, 999],
    labels=["Negative", "Neutral", "Positive", "Elite"]
)

playoff_clean["offense_tier"] = pd.cut(
    playoff_clean["off_rating"],
    bins=[0, 105, 115, 125, 999],
    labels=["Poor", "Average", "Strong", "Elite"]
)

playoff_clean["defense_tier"] = pd.cut(
    playoff_clean["def_rating"],
    bins=[0, 105, 112, 120, 999],
    labels=["Elite", "Strong", "Average", "Poor"]
)

playoff_clean["lineup_stability_score"] = (
    playoff_clean["min"] / playoff_clean["min"].sum()
).round(4)

playoff_clean["two_way_score"] = (
    playoff_clean["off_rating"] - playoff_clean["def_rating"]
).round(2)

## Compare Regular Season vs Playoffs

In [11]:
comparison_cols = [
    "group_name",
    "season_type",
    "min",
    "off_rating",
    "def_rating",
    "net_rating",
    "pace",
    "ast_pct",
    "reb_pct",
    "ts_pct",
    "lineup_quality",
    "offense_tier",
    "defense_tier",
    "lineup_stability_score",
    "two_way_score"
]

In [12]:
regular_compare = regular_df[comparison_cols].copy()
playoff_compare = playoff_clean[comparison_cols].copy()

lineup_comparison = pd.concat(
    [regular_compare, playoff_compare],
    ignore_index=True
)

In [13]:
lineup_comparison.to_csv(
    PROCESSED_DIR / "knicks_lineup_regular_vs_playoffs.csv",
    index=False
)

In [14]:
season_summary = (
    lineup_comparison
    .groupby("season_type")
    .agg(
        lineup_count=("group_name", "count"),
        avg_minutes=("min", "mean"),
        avg_off_rating=("off_rating", "mean"),
        avg_def_rating=("def_rating", "mean"),
        avg_net_rating=("net_rating", "mean"),
        avg_pace=("pace", "mean"),
        avg_ts_pct=("ts_pct", "mean")
    )
    .round(2)
)

season_summary

,lineup_count,avg_minutes,avg_off_rating,avg_def_rating,avg_net_rating,avg_pace,avg_ts_pct
season_type,,,,,,,
Playoffs,14,29.79,117.25,98.06,19.19,99.18,0.62
Regular Season,26,67.81,120.50,110.07,10.45,100.59,0.61


In [15]:
lineup_pivot = (
    lineup_comparison
    .pivot_table(
        index="group_name",
        columns="season_type",
        values=["min", "off_rating", "def_rating", "net_rating"],
        aggfunc="mean"
    )
)

lineup_pivot.head()

def_rating                 \
season_type                                          Playoffs Regular Season   
group_name                                                                     
A. Hukporti - J. Sochan - T. Kolek - P. Dadiet ...      133.3            NaN   
J. Clarkson - J. Hart - M. Bridges - J. Brunson...        NaN          113.0   
J. Clarkson - K. Towns - J. Hart - L. Shamet - ...      105.3            NaN   
J. Clarkson - K. Towns - J. Hart - M. Bridges -...        NaN          127.2   
J. Clarkson - K. Towns - J. Hart - M. Bridges -...        NaN          122.9   

                                                        min                 \
season_type                                        Playoffs Regular Season   
group_name                                                                   
A. Hukporti - J. Sochan - T. Kolek - P. Dadiet ...     11.0            NaN   
J. Clarkson - J. Hart - M. Bridges - J. Brunson...      NaN           36.0   
J. Clarkson - K. Towns - J. Hart - L. Shamet - ...     10.0            NaN   
J. Clarkson - K. Towns - J. Hart - M. Bridges -...      NaN           38.0   
J. Clarkson - K. Towns - J. Hart - M. Bridges -...      NaN           36.0   

                                                   net_rating                 \
season_type                                          Playoffs Regular Season   
group_name                                                                     
A. Hukporti - J. Sochan - T. Kolek - P. Dadiet ...      -67.9            NaN   
J. Clarkson - J. Hart - M. Bridges - J. Brunson...        NaN           25.2   
J. Clarkson - K. Towns - J. Hart - L. Shamet - ...        9.0            NaN   
J. Clarkson - K. Towns - J. Hart - M. Bridges -...        NaN           -8.2   
J. Clarkson - K. Towns - J. Hart - M. Bridges -...        NaN          -15.3   

                                                   off_rating                 
season_type                                          Playoffs Regular Season  
group_name                                                                    
A. Hukporti - J. Sochan - T. Kolek - P. Dadiet ...       65.4            NaN  
J. Clarkson - J. Hart - M. Bridges - J. Brunson...        NaN          138.2  
J. Clarkson - K. Towns - J. Hart - L. Shamet - ...      114.3            NaN  
J. Clarkson - K. Towns - J. Hart - M. Bridges -...        NaN          118.9  
J. Clarkson - K. Towns - J. Hart - M. Bridges -...        NaN          107.6

## Focused Cmomparison on Lineups which appeared in Regular Season and Playoffs

In [16]:
shared_lineups = lineup_pivot.dropna()

shared_lineups.head()

def_rating                 \
season_type                                          Playoffs Regular Season   
group_name                                                                     
J. Hart - M. Bridges - J. Brunson - M. Robinson...      116.3           94.6   
K. Towns - J. Hart - M. Bridges - J. Brunson - ...       63.2          111.0   
K. Towns - J. Hart - M. Bridges - J. Brunson - ...      111.3          116.6   
K. Towns - O. Anunoby - J. Hart - M. Bridges - ...      108.9          112.8   
K. Towns - O. Anunoby - M. Bridges - J. Brunson...       61.5          109.0   

                                                        min                 \
season_type                                        Playoffs Regular Season   
group_name                                                                   
J. Hart - M. Bridges - J. Brunson - M. Robinson...     22.0           41.0   
K. Towns - J. Hart - M. Bridges - J. Brunson - ...     10.0           84.0   
K. Towns - J. Hart - M. Bridges - J. Brunson - ...     32.0          123.0   
K. Towns - O. Anunoby - J. Hart - M. Bridges - ...    205.0          541.0   
K. Towns - O. Anunoby - M. Bridges - J. Brunson...     21.0          119.0   

                                                   net_rating                 \
season_type                                          Playoffs Regular Season   
group_name                                                                     
J. Hart - M. Bridges - J. Brunson - M. Robinson...       14.0           27.7   
K. Towns - J. Hart - M. Bridges - J. Brunson - ...       65.4            0.0   
K. Towns - J. Hart - M. Bridges - J. Brunson - ...        9.3           16.3   
K. Towns - O. Anunoby - J. Hart - M. Bridges - ...        9.9            2.3   
K. Towns - O. Anunoby - M. Bridges - J. Brunson...      101.3           -0.6   

                                                   off_rating                 
season_type                                          Playoffs Regular Season  
group_name                                                                    
J. Hart - M. Bridges - J. Brunson - M. Robinson...      130.2          122.2  
K. Towns - J. Hart - M. Bridges - J. Brunson - ...      128.6          111.0  
K. Towns - J. Hart - M. Bridges - J. Brunson - ...      120.6          132.9  
K. Towns - O. Anunoby - J. Hart - M. Bridges - ...      118.8          115.0  
K. Towns - O. Anunoby - M. Bridges - J. Brunson...      162.8          108.4

In [17]:
shared_lineups["net_rating_delta"] = (
    shared_lineups[("net_rating", "Playoffs")]
    - shared_lineups[("net_rating", "Regular Season")]
)

shared_lineups["off_rating_delta"] = (
    shared_lineups[("off_rating", "Playoffs")]
    - shared_lineups[("off_rating", "Regular Season")]
)

shared_lineups["def_rating_delta"] = (
    shared_lineups[("def_rating", "Playoffs")]
    - shared_lineups[("def_rating", "Regular Season")]
)

shared_lineups["minutes_delta"] = (
    shared_lineups[("min", "Playoffs")]
    - shared_lineups[("min", "Regular Season")]
)

In [18]:
shared_lineups = shared_lineups.reset_index()

## Delta Table

In [27]:
delta_df = shared_lineups.copy()

delta_df["net_rating_delta"] = (
    delta_df[("net_rating", "Playoffs")]
    - delta_df[("net_rating", "Regular Season")]
)

delta_df["off_rating_delta"] = (
    delta_df[("off_rating", "Playoffs")]
    - delta_df[("off_rating", "Regular Season")]
)

delta_df["def_rating_delta"] = (
    delta_df[("def_rating", "Playoffs")]
    - delta_df[("def_rating", "Regular Season")]
)

delta_df = delta_df.reset_index()

delta_df.sort_values(
    "net_rating_delta",
    ascending=False
).head(10)

index                                         group_name  \
season_type                                                            
4               4  K. Towns - O. Anunoby - M. Bridges - J. Brunso...   
1               1  K. Towns - J. Hart - M. Bridges - J. Brunson -...   
3               3  K. Towns - O. Anunoby - J. Hart - M. Bridges -...   
2               2  K. Towns - J. Hart - M. Bridges - J. Brunson -...   
0               0  J. Hart - M. Bridges - J. Brunson - M. Robinso...   
5               5  O. Anunoby - J. Hart - M. Bridges - J. Brunson...   

            def_rating                     min                net_rating  \
season_type   Playoffs Regular Season Playoffs Regular Season   Playoffs   
4                 61.5          109.0     21.0          119.0      101.3   
1                 63.2          111.0     10.0           84.0       65.4   
3                108.9          112.8    205.0          541.0        9.9   
2                111.3          116.6     32.0          123.0        9.3   
0                116.3           94.6     22.0           41.0       14.0   
5                109.4          109.8     13.0           70.0      -12.4   

                           off_rating                net_rating_delta  \
season_type Regular Season   Playoffs Regular Season                    
4                     -0.6      162.8          108.4            101.9   
1                      0.0      128.6          111.0             65.4   
3                      2.3      118.8          115.0              7.6   
2                     16.3      120.6          132.9             -7.0   
0                     27.7      130.2          122.2            -13.7   
5                     24.0       97.0          133.8            -36.4   

            off_rating_delta def_rating_delta minutes_delta  
season_type                                                  
4                       54.4            -47.5         -98.0  
1                       17.6            -47.8         -74.0  
3                        3.8             -3.9        -336.0  
2                      -12.3             -5.3         -91.0  
0                        8.0             21.7         -19.0  
5                      -36.8             -0.4         -57.0

In [28]:
delta_df.to_csv(
    PROCESSED_DIR / "knicks_lineup_playoff_deltas.csv",
    index=False
)

## Playoff Improvements

In [19]:
shared_lineups.sort_values(
    "net_rating_delta",
    ascending=False
)[
    [
        "group_name",
        "net_rating_delta",
        "off_rating_delta",
        "def_rating_delta",
        "minutes_delta"
    ]
].head(10)

,group_name,net_rating_delta,off_rating_delta,def_rating_delta,minutes_delta
season_type,,,,,
4,K. Towns - O. Anunoby - M. Bridges - J. Brunso...,101.9,54.4,-47.5,-98.0
1,K. Towns - J. Hart - M. Bridges - J. Brunson -...,65.4,17.6,-47.8,-74.0
3,K. Towns - O. Anunoby - J. Hart - M. Bridges -...,7.6,3.8,-3.9,-336.0
2,K. Towns - J. Hart - M. Bridges - J. Brunson -...,-7.0,-12.3,-5.3,-91.0
0,J. Hart - M. Bridges - J. Brunson - M. Robinso...,-13.7,8.0,21.7,-19.0
5,O. Anunoby - J. Hart - M. Bridges - J. Brunson...,-36.4,-36.8,-0.4,-57.0


## Playoff Drops

In [20]:
shared_lineups.sort_values(
    "net_rating_delta",
    ascending=True
)[
    [
        "group_name",
        "net_rating_delta",
        "off_rating_delta",
        "def_rating_delta",
        "minutes_delta"
    ]
].head(10)

,group_name,net_rating_delta,off_rating_delta,def_rating_delta,minutes_delta
season_type,,,,,
5,O. Anunoby - J. Hart - M. Bridges - J. Brunson...,-36.4,-36.8,-0.4,-57.0
0,J. Hart - M. Bridges - J. Brunson - M. Robinso...,-13.7,8.0,21.7,-19.0
2,K. Towns - J. Hart - M. Bridges - J. Brunson -...,-7.0,-12.3,-5.3,-91.0
3,K. Towns - O. Anunoby - J. Hart - M. Bridges -...,7.6,3.8,-3.9,-336.0
1,K. Towns - J. Hart - M. Bridges - J. Brunson -...,65.4,17.6,-47.8,-74.0
4,K. Towns - O. Anunoby - M. Bridges - J. Brunso...,101.9,54.4,-47.5,-98.0


In [22]:
shared_lineups_flat = shared_lineups.copy()

shared_lineups_flat.columns = [
    "_".join([str(i) for i in col if i != ""]).strip("_")
    if isinstance(col, tuple)
    else col
    for col in shared_lineups_flat.columns
]

shared_lineups_flat.head()

,group_name,def_rating_Playoffs,def_rating_Regular Season,min_Playoffs,min_Regular Season,net_rating_Playoffs,net_rating_Regular Season,off_rating_Playoffs,off_rating_Regular Season,net_rating_delta,off_rating_delta,def_rating_delta,minutes_delta
0,J. Hart - M. Bridges - J. Brunson - M. Robinso...,116.3,94.6,22.0,41.0,14.0,27.7,130.2,122.2,-13.7,8.0,21.7,-19.0
1,K. Towns - J. Hart - M. Bridges - J. Brunson -...,63.2,111.0,10.0,84.0,65.4,0.0,128.6,111.0,65.4,17.6,-47.8,-74.0
2,K. Towns - J. Hart - M. Bridges - J. Brunson -...,111.3,116.6,32.0,123.0,9.3,16.3,120.6,132.9,-7.0,-12.3,-5.3,-91.0
3,K. Towns - O. Anunoby - J. Hart - M. Bridges -...,108.9,112.8,205.0,541.0,9.9,2.3,118.8,115.0,7.6,3.8,-3.9,-336.0
4,K. Towns - O. Anunoby - M. Bridges - J. Brunso...,61.5,109.0,21.0,119.0,101.3,-0.6,162.8,108.4,101.9,54.4,-47.5,-98.0


In [ ]:


fig = px.scatter(
    shared_lineups_flat,
    x="minutes_delta",
    y="net_rating_delta",
    hover_name="group_name",
    hover_data=[
        "off_rating_delta",
        "def_rating_delta",
        "min_Playoffs",
        "min_Regular Season"
    ],
    title="Playoff Lineup Performance Change",
    template="plotly_dark",
    labels={
        "minutes_delta": "Minutes Change (Playoffs - Regular Season)",
        "net_rating_delta": "Net Rating Change"
    }
)

fig.update_traces(
    marker=dict(
        size=14,
        opacity=0.82,
        line=dict(width=1, color="white")
    )
)

fig.show()

In [25]:
fig.write_html("../assets/playoff_lineup_performance_change.html")

### Playoff Performance Delta Interpretation

This visualization measures how lineup performance changed from the regular season to the playoffs.

Each point represents a lineup combination, with:
- x-axis representing minute changes
- y-axis representing net rating changes

Key observations:
- Some lineups experienced improved performance despite reduced playoff usage.
- Several heavily utilized regular season lineups saw declines in postseason efficiency.
- A subset of playoff lineups demonstrated strong defensive improvements under postseason conditions.

This analysis helps identify lineup survivability and adaptability during playoff basketball.

In [ ]:


fig = px.box(
    lineup_comparison,
    x="season_type",
    y="net_rating",
    title="Regular Season vs Playoff Lineup Net Rating",
    template="plotly_white"
)

fig.show()

In [30]:
fig = px.box(
    lineup_comparison,
    x="season_type",
    y="min",
    title="Regular Season vs Playoff Lineup Usage",
    template="plotly_white"
)

fig.show()

In [31]:
ASSETS_DIR = Path("../assets")
ASSETS_DIR.mkdir(parents=True, exist_ok=True)

fig.write_html(
    ASSETS_DIR / "regular_vs_playoff_lineup_usage.html"
)

### Regular Season vs Playoff Performance Interpretation

This visualization compares lineup performance distributions between the regular season and playoffs.

The analysis highlights how lineup efficiency and usage patterns changed during postseason basketball.

Key observations:
- Playoff rotations became significantly tighter, with fewer lineups receiving meaningful minutes.
- Certain lineup combinations improved defensively during playoff competition.
- Offensive efficiency became more volatile during postseason play.
- High-performing playoff lineups tended to be more concentrated among trusted rotational groups.

This comparison demonstrates how lineup strategy adapts under playoff conditions and increased competitive intensity.